In [6]:
import numpy as np
from math import log, sqrt, exp
from scipy.stats import norm

<div style="background:#fafafa; padding:20px; border-radius:10px;
            border:1px solid #ddd; font-size:110%;">

<b style="font-size:120%;">Black–Scholes PDE and Crank–Nicolson Scheme</b>

<p>
We now turn from the heat equation to the fundamental PDE of option pricing: the
Black–Scholes equation for the value \(V(S,t)\) of a European call option.  
The PDE reads
</p>

$$
V_t
=
\frac{1}{2}\sigma^2 S^2 V_{SS}
+
r S V_S
-
r V,
\qquad
S \in (0, S_{\max}), \ t \in [0,T).
$$

<p>
We solve this equation <b>backwards in time</b> from maturity \(t=T\) to the
present \(t=0\), subject to the usual terminal and boundary conditions for a
European call:
</p>

$$
V(S,T) = \max(S-K,0).
$$

<p>
• Boundary at \(S = 0\):
</p>

$$
V(0,t) = 0,
$$

<p>
since a call on a worthless underlying is worthless.
</p>

<p>
• Boundary at \(S = S_{\max}\):
</p>

$$
V(S_{\max},t) \approx S_{\max} - K e^{-r(T-t)},
$$

<p>
which captures the asymptotic behaviour of a deep in-the-money call.
</p>

<p>
We discretize the spatial domain using a uniform grid
</p>

$$
S_i = i\,\Delta S, \qquad i = 0,1,\dots,N_S,
$$

<p>
and the time interval using
</p>

$$
t^n = n\,\Delta t, \qquad n = 0,1,\dots,N_T.
$$
</div>

In [4]:
class BlackScholesPDECn:
    """
    Crank–Nicolson solver for the Black–Scholes PDE:

        V_t = 0.5 * sigma^2 * S^2 * V_SS + r * S * V_S - r * V

    on S in [0, S_max], t in [0, T], solved backward in time.
    """

    def __init__(self, S0, K, r, sigma, T,
                 S_max_factor=4.0, Ns=200, Nt=200):
        self.S0 = S0
        self.K = K
        self.r = r
        self.sigma = sigma
        self.T = T

        self.S_max = S_max_factor * S0
        self.Ns = Ns
        self.Nt = Nt

        self.dS = self.S_max / Ns
        self.dt = T / Nt

        self.S = np.linspace(0.0, self.S_max, Ns + 1)

    def _setup_matrices(self):
        Ns = self.Ns
        dS = self.dS
        dt = self.dt
        r = self.r
        sigma = self.sigma

        A = np.zeros((Ns - 1, Ns - 1))
        B = np.zeros((Ns - 1, Ns - 1))


        self.a = np.zeros(Ns - 1)
        self.b = np.zeros(Ns - 1)
        self.c = np.zeros(Ns - 1)

        for i in range(1, Ns):
            Si = i * dS

            alpha = 0.5 * sigma**2 * Si**2 / dS**2
            beta = 0.5 * r * Si / dS

            a = dt * (alpha - beta)
            b = dt * (-2.0 * alpha - r)
            c = dt * (alpha + beta)

            self.a[i - 1] = a
            self.b[i - 1] = b
            self.c[i - 1] = c

            # A = I - 0.5 * dt * L
            if i > 1:
                A[i - 1, i - 2] = -0.5 * a
            A[i - 1, i - 1] = 1.0 - 0.5 * b
            if i < Ns - 1:
                A[i - 1, i] = -0.5 * c

            # B = I + 0.5 * dt * L
            if i > 1:
                B[i - 1, i - 2] = 0.5 * a
            B[i - 1, i - 1] = 1.0 + 0.5 * b
            if i < Ns - 1:
                B[i - 1, i] = 0.5 * c

        return A, B

    def solve_grid(self):
        Ns, Nt = self.Ns, self.Nt
        dt = self.dt
        r = self.r
        K = self.K

        S = self.S
        V = np.maximum(S - K, 0.0)

        A, B = self._setup_matrices()

        for n in range(Nt):
            t_next = self.T - (n + 1) * dt

            V_inner = V[1:Ns]
            rhs = B @ V_inner


            V_0 = 0.0
            V_Smax = self.S_max - K * np.exp(-r * t_next)
            rhs[-1] += 0.5 * self.c[-1] * V_Smax

            V_new_inner = np.linalg.solve(A, rhs)

            V[0] = V_0
            V[1:Ns] = V_new_inner
            V[Ns] = V_Smax

        return V

    def price(self):
        V = self.solve_grid()
        return np.interp(self.S0, self.S, V)

In [5]:
S0 = 100
K = 100
r = 0.05
sigma = 0.2
T = 1.0

solver = BlackScholesPDECn(S0, K, r, sigma, T, Ns=200, Nt=200)
bs_pde_price = solver.price()
bs_pde_price

10.440691506307738

In [7]:
def bs_call_price(S0, K, r, sigma, T):
    d1 = (log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return S0*norm.cdf(d1) - K*exp(-r*T)*norm.cdf(d2)

pde_price = solver.price()
bs_price = bs_call_price(S0, K, r, sigma, T)

print(f"PDE price:      {pde_price:.6f}")
print(f"BS analytic:    {bs_price:.6f}")
print(f"absolute error: {abs(pde_price - bs_price):.6f}")

PDE price:      10.440692
BS analytic:    10.450584
absolute error: 0.009892
